# 1D-CNN Time-Series Model for Harsh Driving Event Detection

This notebook implements a 1D Convolutional Neural Network (CNN) in PyTorch to classify windowed IMU sequence data into driving behavior categories (e.g., sudden acceleration, sudden braking, harsh turns, harsh lane changes, and safe driving).

### Improvements Over Previous Models:
1. **Causal Real-time Architecture:** Uses pure 1D-CNN blocks rather than Bidirectional LSTMs, which avoids lookahead latency and is suitable for streaming real-time inference.
2. **Feature Standardization:** Standardizes sequence features across time and samples using `StandardScaler` fitted on the training set.
3. **Dataset Balancing:** Applies `RandomOverSampler` on the flattened sequence train set to balance minority classes.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
import joblib
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
ROOT_DIR = Path("..")
DATASET_DIR = ROOT_DIR / "dataset"
ARTIFACT_DIR = ROOT_DIR / "artifacts"

sys.path.append(str(ROOT_DIR / "src"))
from harsh_event_model import load_row_labeled_data, IMU_COLUMNS as imu_cols

## 1. Load Data
Load row-level labeled IMU datasets.

In [ ]:
print("Loading raw labeled data...")
raw_df = load_row_labeled_data(DATASET_DIR)
print("Data shape:", raw_df.shape)
print("Label distribution:\n", raw_df["event_label"].value_counts())

## 2. Apply Light Denoising
Perform rolling median filtering over a short window size of 3.

In [ ]:
print("Applying denoising rolling median...")
clean_df = raw_df.copy()
for col in imu_cols:
    clean_df[col] = (
        clean_df.groupby('source_file')[col]
        .transform(lambda s: s.rolling(window=3, min_periods=1, center=True).median())
    )

## 3. Feature Engineering
Compute calculated stream features (accelerometer/gyroscope magnitudes, jerk, energy ratios, yaw vs roll/pitch relations, etc.).

In [ ]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Magnitudes
    df['acc_mag'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)
    df['gyro_mag'] = np.sqrt(df['gyro_x']**2 + df['gyro_y']**2 + df['gyro_z']**2)
    df['mag_mag'] = np.sqrt(df['mag_x']**2 + df['mag_y']**2 + df['mag_z']**2)
    
    # Jerk
    for axis in ['acc_x', 'acc_y', 'acc_z']:
        df[f'{axis}_jerk'] = df.groupby('source_file')[axis].diff().fillna(0.0)
        
    # Ratios & Absolutes
    df['lat_long_ratio'] = df['acc_y'].abs() / (df['acc_x'].abs() + 1e-6)
    df['acc_x_abs'] = df['acc_x'].abs()
    df['acc_y_abs'] = df['acc_y'].abs()
    df['gyro_z_abs'] = df['gyro_z'].abs()
    
    # Energies
    df['acc_lat_energy'] = df['acc_y'] ** 2
    df['acc_long_energy'] = df['acc_x'] ** 2
    df['gyro_yaw_energy'] = df['gyro_z'] ** 2
    
    # Ratios & interactions
    df['lat_long_energy_ratio'] = df['acc_y_abs'] / (df['acc_x_abs'] + 1e-6)
    df['turn_vs_lateral'] = df['gyro_z_abs'] / (df['acc_y_abs'] + 1e-6)
    df['yaw_acc_corr'] = df['gyro_z'] * df['acc_y']
    
    # Diffs
    df['gyro_z_diff'] = df.groupby('source_file')['gyro_z'].diff().fillna(0.0)
    df['acc_y_diff'] = df.groupby('source_file')['acc_y'].diff().fillna(0.0)
    
    # Rolling stats
    df['gyro_z_roll_std5'] = df.groupby('source_file')['gyro_z'].transform(lambda s: s.rolling(5, min_periods=1).std().fillna(0.0))
    df['acc_y_roll_std5'] = df.groupby('source_file')['acc_y'].transform(lambda s: s.rolling(5, min_periods=1).std().fillna(0.0))
    df['acc_y_roll_mean5'] = df.groupby('source_file')['acc_y'].transform(lambda s: s.rolling(5, min_periods=1).mean().fillna(0.0))
    
    # Sign change
    df['gyro_z_sign_change'] = df.groupby('source_file')['gyro_z'].transform(
        lambda s: (s.shift(1) * s < 0).astype(int).fillna(0)
    )
    
    # Roll/pitch turn features
    df['gyro_x_abs'] = df['gyro_x'].abs()
    df['gyro_y_abs'] = df['gyro_y'].abs()
    df['gyro_x_energy'] = df['gyro_x'] ** 2
    df['gyro_y_energy'] = df['gyro_y'] ** 2
    df['gyro_roll_pitch_mag'] = np.sqrt(df['gyro_x']**2 + df['gyro_y']**2)
    df['gyro_total_mag'] = np.sqrt(df['gyro_x']**2 + df['gyro_y']**2 + df['gyro_z']**2)
    df['yaw_vs_roll_pitch'] = df['gyro_z_abs'] / (df['gyro_roll_pitch_mag'] + 1e-6)
    df['gyro_x_roll_std5'] = df.groupby('source_file')['gyro_x'].transform(lambda s: s.rolling(5, min_periods=1).std().fillna(0.0))
    df['gyro_y_roll_std5'] = df.groupby('source_file')['gyro_y'].transform(lambda s: s.rolling(5, min_periods=1).std().fillna(0.0))
    
    return df

clean_df = add_engineered_features(clean_df)

feature_cols_stream = imu_cols + [
    'acc_mag', 'gyro_mag', 'mag_mag',
    'acc_x_jerk', 'acc_y_jerk', 'acc_z_jerk',
    'lat_long_ratio',
    'acc_x_abs', 'acc_y_abs', 'gyro_z_abs',
    'acc_lat_energy', 'acc_long_energy', 'gyro_yaw_energy',
    'lat_long_energy_ratio', 'turn_vs_lateral', 'yaw_acc_corr',
    'gyro_z_diff', 'acc_y_diff',
    'gyro_z_roll_std5', 'acc_y_roll_std5', 'acc_y_roll_mean5',
    'gyro_z_sign_change',
    'gyro_x_abs', 'gyro_y_abs', 'gyro_x_energy', 'gyro_y_energy',
    'gyro_roll_pitch_mag', 'gyro_total_mag', 'yaw_vs_roll_pitch',
    'gyro_x_roll_std5', 'gyro_y_roll_std5',
]
print("Feature stream dimensions:", len(feature_cols_stream))

## 4. Sliding Window Segmentation
Segment dataframes into sliding sequences of size 25 with step size 5.

In [ ]:
WINDOW_SIZE = 25
STEP_SIZE = 5
MIN_EVENT_RATIO = 0.20

def build_window_sequences(df: pd.DataFrame, feature_cols, window_size=25, step_size=5, min_event_ratio=0.20):
    x_seq, y, groups = [], [], []
    for file_name, g in df.groupby('source_file'):
        g = g.reset_index(drop=True)
        if len(g) < window_size:
            continue
        values = g[feature_cols].to_numpy(dtype=np.float32)
        labels = g['event_label'].to_numpy()

        for start in range(0, len(g) - window_size + 1, step_size):
            end = start + window_size
            w = values[start:end]
            wl = labels[start:end]
            non_safe = wl[wl != 'safe']
            if len(non_safe) == 0 or (len(non_safe) / window_size) < min_event_ratio:
                target = 'safe'
            else:
                target = Counter(non_safe).most_common(1)[0][0]

            x_seq.append(w)
            y.append(target)
            groups.append(file_name)

    x_seq = np.stack(x_seq).astype(np.float32)
    y = np.array(y)
    groups = np.array(groups)
    return x_seq, y, groups

x_seq, y, groups = build_window_sequences(clean_df, feature_cols_stream, WINDOW_SIZE, STEP_SIZE, MIN_EVENT_RATIO)
print("Sequences shape:", x_seq.shape)
print("Window label distribution:", Counter(y))

## 5. Train / Val / Test Partitioning
Use `split_manifest.csv` to ensure consistent file-grouped split sets.

In [ ]:
split_manifest_path = ARTIFACT_DIR / 'split_manifest.csv'
split_manifest = pd.read_csv(split_manifest_path)
train_files = split_manifest[split_manifest['split'] == 'train']['group_file'].unique()
val_files = split_manifest[split_manifest['split'] == 'val']['group_file'].unique()
test_files = split_manifest[split_manifest['split'] == 'test']['group_file'].unique()

train_mask = np.isin(groups, train_files)
val_mask = np.isin(groups, val_files)
test_mask = np.isin(groups, test_files)

x_seq_train, y_train_raw = x_seq[train_mask], y[train_mask]
x_seq_val, y_val_raw = x_seq[val_mask], y[val_mask]
x_seq_test, y_test_raw = x_seq[test_mask], y[test_mask]

print(f"Split sizes | Train: {len(x_seq_train)} | Val: {len(x_seq_val)} | Test: {len(x_seq_test)}")

## 6. Target Encoding & Sequence Standardization

In [ ]:
label_order = sorted(np.unique(y).tolist())
label_to_idx = {c: i for i, c in enumerate(label_order)}
idx_to_label = {i: c for c, i in label_to_idx.items()}

y_train = np.array([label_to_idx[v] for v in y_train_raw], dtype=np.int64)
y_val = np.array([label_to_idx[v] for v in y_val_raw], dtype=np.int64)
y_test = np.array([label_to_idx[v] for v in y_test_raw], dtype=np.int64)

# Standardize sequence features using StandardScaler fitted on train flat data
scaler = StandardScaler()
B_train, T, F = x_seq_train.shape
scaler.fit(x_seq_train.reshape(-1, F))

x_seq_train = scaler.transform(x_seq_train.reshape(-1, F)).reshape(B_train, T, F)
x_seq_val = scaler.transform(x_seq_val.reshape(-1, F)).reshape(x_seq_val.shape[0], T, F)
x_seq_test = scaler.transform(x_seq_test.reshape(-1, F)).reshape(x_seq_test.shape[0], T, F)

## 7. Class Balancing (Oversampling)
Oversample the sequences using the 3D-to-2D flattening trick.

In [ ]:
ros = RandomOverSampler(random_state=SEED)
x_seq_train_flat = x_seq_train.reshape(B_train, -1)
x_seq_train_bal_flat, y_train_bal = ros.fit_resample(x_seq_train_flat, y_train)
x_seq_train_bal = x_seq_train_bal_flat.reshape(-1, T, F)

print("Oversampled train class counts:", np.bincount(y_train_bal))

## 8. PyTorch Datasets & Model Architecture

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = SeqDataset(x_seq_train_bal, y_train_bal)
val_ds = SeqDataset(x_seq_val, y_val)
test_ds = SeqDataset(x_seq_test, y_test)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

class TimeSeriesCNN1D(nn.Module):
    def __init__(self, in_features, n_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=in_features, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = x.transpose(1, 2) # [B, F, T]
        x = self.conv(x)
        pooled = x.mean(dim=2) # [B, 256]
        return self.head(pooled)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TimeSeriesCNN1D(in_features=F, n_classes=len(label_order)).to(device)

## 9. Training with Early Stopping & Cosine Annealing LR Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
    avg_loss = total_loss / max(len(loader.dataset), 1)
    from sklearn.metrics import f1_score
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, macro_f1, np.array(all_labels), np.array(all_preds)

EPOCHS = 35
best_val_f1 = -1.0
best_state = None
PATIENCE = 10
patience_count = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * xb.size(0)
        
    scheduler.step()
    train_loss = train_loss_sum / len(train_loader.dataset)
    val_loss, val_f1, _, _ = evaluate(model, val_loader)
    
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1={val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_count = 0
    else:
        patience_count += 1
        
    if patience_count >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

if best_state is not None:
    model.load_state_dict(best_state)

## 10. Evaluation & Visualizations

In [ ]:
test_loss, test_f1, y_true_test, y_pred_test = evaluate(model, test_loader)
print("\n--- Test Classification Report ---")
print(classification_report(y_true_test, y_pred_test, labels=np.arange(len(label_order)), target_names=label_order, digits=4, zero_division=0))

cm = confusion_matrix(y_true_test, y_pred_test, labels=np.arange(len(label_order)))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_order, yticklabels=label_order)
plt.title('Test Set Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

## 11. Save CNN Model Bundle
Save model parameters, scaler, features, and configuration to be loaded by the real-time simulation detector.

In [ ]:
bundle_path = ARTIFACT_DIR / "harsh_event_cnn_bundle.pth"
bundle = {
    "model_state_dict": best_state if best_state is not None else model.state_dict(),
    "scaler": scaler,
    "label_encoder_classes": label_order,
    "feature_cols_stream": feature_cols_stream,
    "window_size": WINDOW_SIZE,
    "step_size": STEP_SIZE,
    "min_event_ratio": MIN_EVENT_RATIO,
    "in_features": F,
    "n_classes": len(label_order),
}
torch.save(bundle, bundle_path)
print("Saved bundle to:", bundle_path)